# Reshaping tables: crosstab, melt, pivot and pivot_table

The same data can be stored as a long table or as a wide table. After this block you can say which shape a question needs, move a table between the two shapes with `melt`, `pivot` and `pivot_table`, count pairs of categories with `crosstab`, and predict the shape of the result before you run the line.

## Overview

| Section | What it covers |
|---|---|
| Long and wide tables | the two shapes of the same data, and which tool goes which way |
| crosstab: counting combinations | counts of category pairs, totals, shares, other summaries |
| melt: wide to long | column names become values; the four parameters |
| pivot: long to wide | values become column names; the duplicates rule; the round trip |
| pivot_table: reshape and aggregate | pivot that combines duplicates; options; pivot_table as a groupby |
| Worked example · Room types across the Berlin districts | the four tools on the 10,221 real listings |
| Check your understanding | four questions to vote on |
| Exercises | practice in pairs |
| Debrief | what to compare |
| Further reading | the pandas guide and the books |

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

In [2]:
import pandas as pd   # pandas: tables with labelled rows and columns

## Long and wide tables

The same information can be stored in two shapes.

- **Long**: one row per observation, and a column that says which category the observation belongs to. Long tables are easy to filter, to group and to plot.
- **Wide**: one row per item, and one column per category. Wide tables are easy for people to read and compare.

All four tools of this block reshape a table, and they come in pairs: `melt` turns wide data into long data; `pivot` and `pivot_table` turn long data into wide data. `crosstab` is a shortcut for one very common wide table: the counts of category pairs. So `crosstab` and `melt` are not opposites; the opposite pair is `melt` and `pivot`.

| Tool | Direction | Does maths? | In one line |
|---|---|---|---|
| `pd.crosstab` | long → wide | yes, counts by default | How many rows have each combination? |
| `df.melt` | wide → long | no | Turn column names into values |
| `df.pivot` | long → wide | no | Turn values into column names |
| `df.pivot_table` | long → wide | yes, the mean by default | pivot, and aggregate duplicates |

Every example below uses one small table of made-up Berlin listings, so you can see every row.

In [3]:
# Six made-up listings: one row per listing (a long table)
listings = pd.DataFrame({
    "listing_id": [1, 2, 3, 4, 5, 6],
    "district":   ["Mitte", "Mitte", "Mitte", "Kreuzberg", "Kreuzberg", "Pankow"],
    "room_type":  ["Entire home", "Private room", "Entire home",
                   "Private room", "Entire home", "Private room"],
    "price":      [120, 55, 140, 45, 95, 40],   # EUR per night
})
listings

,listing_id,district,room_type,price
0,1,Mitte,Entire home,120
1,2,Mitte,Private room,55
2,3,Mitte,Entire home,140
3,4,Kreuzberg,Private room,45
4,5,Kreuzberg,Entire home,95
5,6,Pankow,Private room,40


Notice the two entire homes in Mitte, 120 and 140 EUR. They are the pair that later decides what `pivot` can and cannot do.

> **Quiz.** You want a table a manager can read at a glance: districts down the side, room types across the top. Is that the long or the wide shape?

## crosstab: counting combinations

`pd.crosstab` answers one question: **how many rows have this category and that category?** The first argument becomes the rows, the second becomes the columns, and each cell is a count. It takes two Series, not column names, which is why it is called as `pd.crosstab(...)` and not as `listings.crosstab(...)`.

**Predict.** How many cells will the result have, and what number will stand in the cell Pankow, Entire home?

In [4]:
# pd.crosstab(rows, columns): the values of the first Series become the rows,
# the values of the second the columns; each cell counts the rows with that pair
pd.crosstab(listings["district"], listings["room_type"])

room_type,Entire home,Private room
district,,
Kreuzberg,1,1
Mitte,2,1
Pankow,0,1


Read a cell like a sentence: Mitte has **2** entire homes. Pankow has **0** entire homes, because no row has that combination. Three districts and two room types give 3 × 2 = 6 cells.

### Totals with margins

`margins=True` adds a row and a column called `All` with the totals.

In [5]:
# margins=True: add an "All" row and an "All" column with the totals
pd.crosstab(listings["district"], listings["room_type"], margins=True)

room_type,Entire home,Private room,All
district,,,
Kreuzberg,1,1,2
Mitte,2,1,3
Pankow,0,1,1
All,3,3,6


### Shares with normalize

`normalize=` divides the counts so that they become shares. Its value says which part of the table sums to 1:

- `normalize="index"`: each **row** sums to 1. "What share of Mitte's listings are entire homes?"
- `normalize="columns"`: each **column** sums to 1. "What share of all entire homes are in Mitte?"
- `normalize="all"`: the **whole table** sums to 1.

**Predict.** With `normalize="index"`, what are the two numbers in the Pankow row?

In [6]:
# normalize="index": every row sums to 1; .round(2) keeps two decimals
pd.crosstab(listings["district"], listings["room_type"], normalize="index").round(2)

room_type,Entire home,Private room
district,,
Kreuzberg,0.50,0.50
Mitte,0.67,0.33
Pankow,0.00,1.00


Pankow has one listing, a private room, so its row is 0.00 and 1.00. Mitte's row says that 67 % of Mitte's listings are entire homes.

In [7]:
# normalize="columns": every column sums to 1 (share of each room type per district)
pd.crosstab(listings["district"], listings["room_type"], normalize="columns").round(2)

room_type,Entire home,Private room
district,,
Kreuzberg,0.33,0.33
Mitte,0.67,0.33
Pankow,0.00,0.33


### Other summaries with values and aggfunc

A count is only the default. Pass **both** `values=` (a number column) and `aggfunc=` (a summary such as `"mean"`, `"sum"` or `"max"`), and each cell summarises that column for its pair instead of counting rows.

In [8]:
# values= the column to summarise, aggfunc= how to summarise it
pd.crosstab(listings["district"], listings["room_type"],
            values=listings["price"], aggfunc="mean")

room_type,Entire home,Private room
district,,
Kreuzberg,95.0,45.0
Mitte,130.0,55.0
Pankow,NaN,40.0


Mitte's two entire homes cost 120 and 140 EUR, so the mean is 130. Pankow has no entire home, so that cell is `NaN`, not 0: there is nothing to average.

The next cell passes `values=` without `aggfunc=` on purpose, to show the error.

In [9]:
try:
    pd.crosstab(listings["district"], listings["room_type"], values=listings["price"])
except ValueError as error:
    print("ValueError:", error)

ValueError: values cannot be used without an aggfunc.


Read the message: `values cannot be used without an aggfunc`. pandas does not guess how to summarise the prices; you say it with `aggfunc=`.

## melt: wide to long

`melt` **unpivots**: it takes several columns and stacks them into two columns, one holding the old column **names** and one holding their **values**. No maths happens; every number is kept.

Here is a wide table: the average price per district, one column per room type.

In [10]:
# A wide table: one row per district, one column per room type
wide = pd.DataFrame({
    "district":     ["Mitte", "Kreuzberg"],
    "Entire home":  [130, 95],
    "Private room": [55, 45],
})
wide

,district,Entire home,Private room
0,Mitte,130,55
1,Kreuzberg,95,45


**Predict.** After `melt`, how many rows and how many columns will the table have?

In [11]:
long = wide.melt(
    id_vars="district",       # columns to keep as they are; they repeat on every row
    var_name="room_type",     # new column for the old column NAMES
    value_name="avg_price",   # new column for the VALUES
)
long

,district,room_type,avg_price
0,Mitte,Entire home,130
1,Kreuzberg,Entire home,95
2,Mitte,Private room,55
3,Kreuzberg,Private room,45


The headers "Entire home" and "Private room" became **values** of the column `room_type`. The table went from 2 rows × 3 columns to 4 rows × 3 columns: the same data, in a taller shape. In general: rows after = rows before × number of melted columns, and columns after = id_vars + 2.

### The four parameters of melt

| Parameter | What it does | Default |
|---|---|---|
| `id_vars` | the columns that stay and repeat on every row | none |
| `value_vars` | the columns to melt | all columns not in `id_vars` |
| `var_name` | the name of the column that holds the old column names | `"variable"` |
| `value_name` | the name of the column that holds the values | `"value"` |

Without `var_name` and `value_name` you get the generic names:

In [12]:
# Only id_vars: the new columns are called "variable" and "value"
wide.melt(id_vars="district")

,district,variable,value
0,Mitte,Entire home,130
1,Kreuzberg,Entire home,95
2,Mitte,Private room,55
3,Kreuzberg,Private room,45


### Melting only some columns

To melt only some columns, list them in `value_vars`. Columns that are in neither list are dropped.

In [13]:
# value_vars: melt only "Entire home"; "Private room" is dropped
wide.melt(id_vars="district", value_vars=["Entire home"])

,district,variable,value
0,Mitte,Entire home,130
1,Kreuzberg,Entire home,95


### Why long data helps

- **Plotting libraries** such as seaborn expect long data: one column for x, one for y, one for the colour.
- **groupby and filtering** are easier when a category lives in one column instead of across many.
- **Spreadsheets** often have one column per year (`2023`, `2024`, `2025`); `melt` turns them into `year | value`.

## pivot: long to wide

`pivot` is the exact opposite of `melt`: it turns the **values** of one column into **column headers**. It does no maths, so it works only when every pair of row and column appears **exactly once**.

Here is a long table: the price per night of two listings on two dates.

In [14]:
# A long table: one row per date and listing
daily = pd.DataFrame({
    "date":       ["11-01", "11-01", "11-02", "11-02"],
    "listing_id": ["A", "B", "A", "B"],
    "price":      [80, 120, 85, 125],
})
daily

,date,listing_id,price
0,11-01,A,80
1,11-01,B,120
2,11-02,A,85
3,11-02,B,125


In [15]:
# index=   each unique date becomes a ROW
# columns= each unique listing becomes a COLUMN
# values=  the prices fill the CELLS
daily.pivot(index="date", columns="listing_id", values="price")

listing_id,A,B
date,,
11-01,80,120
11-02,85,125


The labels `listing_id` and `date` in the corner are the names of the two axes. To get a plain flat table back, move the index into a column and remove the name of the columns axis:

In [16]:
# reset_index(): the dates go back into a column
# rename_axis(None, axis=1): remove the name "listing_id" above the columns
daily.pivot(index="date", columns="listing_id", values="price").reset_index().rename_axis(None, axis=1)

,date,A,B
0,11-01,80,120
1,11-02,85,125


### The duplicates rule of pivot

Add a second price for listing A on 11-01 (80 and 90). **Predict.** What will `pivot` do now: average the two, keep the first, or stop?

The next cell fails on purpose and prints the error.

In [17]:
# dup: the table daily with one more row for 11-01 and A
dup = pd.concat([daily, pd.DataFrame({"date": ["11-01"], "listing_id": ["A"], "price": [90]})],
                ignore_index=True)   # ignore_index=True: number the rows 0, 1, 2, ... again
try:
    dup.pivot(index="date", columns="listing_id", values="price")
except ValueError as error:
    print("ValueError:", error)

ValueError: Index contains duplicate entries, cannot reshape


Read the message: `Index contains duplicate entries, cannot reshape`. The cell (11-01, A) would need two values, and `pivot` refuses to choose. That strictness is useful: it warns you about duplicates you did not expect. Check for them first:

In [18]:
# duplicated(subset=[...]): True for a row whose pair already appeared above it
# .any(): True if at least one row is a duplicate, so pivot would fail
dup.duplicated(subset=["date", "listing_id"]).any()

np.True_

### The round trip of melt and pivot

Pivoting the melted table from the melt section gives the wide table back: `index=` the id_vars, `columns=` the var_name, `values=` the value_name.

In [19]:
# melt = columns -> values; pivot = values -> columns: each undoes the other
long.pivot(index="district", columns="room_type", values="avg_price")

room_type,Entire home,Private room
district,,
Kreuzberg,95,45
Mitte,130,55


The same four prices. `pivot` sorts the rows, so Kreuzberg now comes before Mitte.

## pivot_table: reshape and aggregate

`pivot_table` is `pivot` that can handle duplicates: when several rows land in the same cell, it **combines** them with `aggfunc`, the **mean** by default. It always works, which makes it the safe choice when you are not sure the pairs are unique.

### pivot_table on duplicates

**Predict.** On `dup`, which number stands in the cell (11-01, A)?

In [20]:
# Same arguments as pivot; the rows of one cell are combined, with the mean by default
dup.pivot_table(index="date", columns="listing_id", values="price")

listing_id,A,B
date,,
11-01,85.0,120.0
11-02,85.0,125.0


(80 + 90) / 2 = 85. Choose another summary with `aggfunc=`:

In [21]:
# aggfunc="sum": add the two prices instead of averaging them
dup.pivot_table(index="date", columns="listing_id", values="price", aggfunc="sum")

listing_id,A,B
date,,
11-01,170,120
11-02,85,125


On the listings, Mitte has two entire homes, so `pivot` would fail and `pivot_table` gives the mean:

In [22]:
listings.pivot_table(index="district", columns="room_type", values="price")

room_type,Entire home,Private room
district,,
Kreuzberg,95.0,45.0
Mitte,130.0,55.0
Pankow,NaN,40.0


This is the same table as the `crosstab` with `values=` and `aggfunc="mean"` above: the same result, a different way to call it.

### Margins and fill_value

`margins=True` adds an `All` row and column; `fill_value=0` writes 0 where a pair has no rows.

In [23]:
listings.pivot_table(
    index="district", columns="room_type", values="price",
    aggfunc="mean",
    margins=True,     # adds an "All" row and an "All" column
    fill_value=0,     # empty combinations show 0 instead of NaN
).round(1)            # one decimal is enough for prices

room_type,Entire home,Private room,All
district,,,
Kreuzberg,95.0,45.0,70.0
Mitte,130.0,55.0,105.0
Pankow,0.0,40.0,40.0
All,118.3,46.7,82.5


Careful: `fill_value=0` makes Pankow's entire-home price look like €0. That is fine for counts or sums, but misleading for averages.

### Several statistics at once

A list in `aggfunc=` gives one column per summary. Without `columns=`, there is one row per district.

In [24]:
listings.pivot_table(index="district", values="price", aggfunc=["mean", "max", "count"])

,mean,max,count
,price,price,price
district,,,
Kreuzberg,70.0,95,2
Mitte,105.0,140,3
Pankow,40.0,40,1


### pivot_table as a groupby

These two lines give exactly the same table. `index=` and `columns=` are the two groupby keys, `values=` is the column you summarise, `aggfunc=` is the function, and `unstack()` moves the second key into the columns.

In [25]:
via_pivot_table = listings.pivot_table(index="district", columns="room_type", values="price")
# groupby by both keys, mean of price, then move room_type from the rows into the columns
via_groupby = listings.groupby(["district", "room_type"])["price"].mean().unstack()
via_pivot_table.equals(via_groupby)   # True: the same values, labels and shape

True

### pivot_table or crosstab

Both build a long → wide grid of summarised cells, and `crosstab` calls `pivot_table` internally. The differences are convenience:

| | `crosstab` | `pivot_table` |
|---|---|---|
| Default cell | count | mean |
| Input | Series: `pd.crosstab(s1, s2)` | column names: `df.pivot_table(index=..., columns=...)` |
| Shares | built in: `normalize=` | divide yourself |
| Data from different DataFrames or arrays | yes | no, one DataFrame |

Pick the one whose default matches your question: **how many** per district and room type → `crosstab`; **average price** per district and room type → `pivot_table`.

### Which tool do I need?

1. The data is wide and you want it long (for seaborn, groupby, tidy data) → `melt`.
2. You want counts or shares of two categories → `crosstab`.
3. You want an average, sum or maximum of a number per category pair → `pivot_table`.
4. The data is long, one row per combination, and you only want it wide → `pivot`.
5. You are not sure whether pairs repeat → `pivot_table`. It never fails.

## Worked example · Room types across the Berlin districts

The same four tools on the real Inside Airbnb listings. To keep the names apart, the real table is called `listings_berlin`, as in notebook 03.

### Step 1 · Load the listings

In [26]:
listings_berlin = pd.read_parquet("case-study/data/airbnb/listings.parquet")   # 10,221 listings
listings_berlin[["district", "room_type", "price"]].head()

,district,room_type,price
0,Pankow,Entire home/apt,160.71
1,Pankow,Entire home/apt,193.33
2,Pankow,Entire home/apt,NaN
3,Friedrichshain-Kreuzberg,Entire home/apt,10.88
4,Pankow,Entire home/apt,372.67


### Step 2 · Count room types per district

**Why this step?** Before you compare prices, check how many listings stand behind each cell: a mean of three listings says little.

In [27]:
room_counts = pd.crosstab(listings_berlin["district"], listings_berlin["room_type"], margins=True)
room_counts

room_type,Entire home/apt,Hotel room,Private room,Shared room,All
district,,,,,
Charlottenburg-Wilm.,770,14,362,0,1146
Friedrichshain-Kreuzberg,1429,4,663,23,2119
Lichtenberg,200,9,108,1,318
Marzahn - Hellersdorf,61,0,34,0,95
Mitte,1628,23,582,35,2268
Neukölln,637,0,404,5,1046
Pankow,1215,15,329,3,1562
Reinickendorf,77,0,62,2,141
Spandau,67,3,37,0,107


### Step 3 · Shares within each district

In [28]:
# normalize="index": the mix of room types inside each district; sort by the share of private rooms
room_shares = pd.crosstab(listings_berlin["district"], listings_berlin["room_type"], normalize="index")
room_shares.sort_values("Private room").round(2)

room_type,Entire home/apt,Hotel room,Private room,Shared room
district,,,,
Pankow,0.78,0.01,0.21,0.00
Treptow - Köpenick,0.75,0.00,0.25,0.00
Mitte,0.72,0.01,0.26,0.02
Friedrichshain-Kreuzberg,0.67,0.00,0.31,0.01
Charlottenburg-Wilm.,0.67,0.01,0.32,0.00
Steglitz - Zehlendorf,0.68,0.00,0.32,0.00
Tempelhof - Schöneberg,0.67,0.01,0.32,0.00
Lichtenberg,0.63,0.03,0.34,0.00
Spandau,0.63,0.03,0.35,0.00


### Step 4 · Median price per district and room type

The listings have many duplicates per pair, so `pivot` would fail; `pivot_table` with `aggfunc="median"` gives a price that a few very expensive listings cannot pull up.

In [29]:
median_prices = listings_berlin.pivot_table(index="district", columns="room_type",
                                            values="price", aggfunc="median")
median_prices[["Entire home/apt", "Private room"]].sort_values("Entire home/apt").round(0)

room_type,Entire home/apt,Private room
district,,
Reinickendorf,100.0,71.0
Marzahn - Hellersdorf,113.0,62.0
Treptow - Köpenick,125.0,63.0
Charlottenburg-Wilm.,130.0,93.0
Steglitz - Zehlendorf,130.0,80.0
Tempelhof - Schöneberg,133.0,83.0
Neukölln,144.0,92.0
Lichtenberg,153.0,80.0
Spandau,160.0,76.0


### Step 5 · Back to long for plotting

A plotting library wants one row per district and room type. `melt` turns the wide table back into a long one; `reset_index()` first moves the districts into a column, so they can be an id_var.

In [30]:
median_long = (median_prices[["Entire home/apt", "Private room"]]
               .reset_index()                                   # district becomes a column
               .melt(id_vars="district", var_name="room_type", value_name="median_price"))
median_long.head()

,district,room_type,median_price
0,Charlottenburg-Wilm.,Entire home/apt,129.50
1,Friedrichshain-Kreuzberg,Entire home/apt,165.00
2,Lichtenberg,Entire home/apt,152.67
3,Marzahn - Hellersdorf,Entire home/apt,113.00
4,Mitte,Entire home/apt,172.75


> **Quiz.** `median_long` has how many rows? Work it out from the shape of `median_prices` before you run `len(median_long)`.

## Check your understanding

Vote alone first, then discuss with your neighbour, then vote again.

1. `pd.crosstab(listings["district"], listings["room_type"], normalize="columns")`: which value stands in the cell Kreuzberg, Private room? **A** 0.33 · **B** 0.50 · **C** 0.17 · **D** 1
2. `reviews` has the columns `district`, `2024` and `2025` and two rows. `reviews.melt(id_vars="district")` has which shape? **A** (4, 3) · **B** (2, 3) · **C** (4, 2) · **D** (6, 3)
3. `listings.pivot(index="district", columns="room_type", values="price")` gives: **A** the mean for Mitte's entire homes · **B** the first price, 120 · **C** a ValueError · **D** NaN for Mitte's entire homes
4. Which call can produce shares that sum to 1 in each row without any extra division? **A** `pivot` · **B** `pivot_table` · **C** `crosstab` · **D** `melt`

## Exercises

Work in pairs; the lecturer picks which exercises to set. All use the `listings` table from the top unless they say otherwise.

### 4.1 · Where are the entire homes?

*Code*

> What share of all entire homes are in Mitte? Write one `crosstab` call that answers it, then say in one sentence what the number means.

*Source: course exercise.*

### 4.2 · Reviews per year, long

*Code*

> Turn this table into long format with the columns `district | year | reviews`:
>
> ```python
> reviews = pd.DataFrame({"district": ["Mitte", "Pankow"],
>                         "2024": [310, 90], "2025": [420, 130]})
> ```

*Source: course exercise.*

### 4.3 · And back again

*Code*

> Turn your answer to 4.2 back into the original wide table. Which tool can you use, and why is it safe here?

*Source: course exercise.*

### 4.4 · The most expensive listing per pair

*Code*

> Show the **highest** price per district and room type.

*Source: course exercise.*

### 4.5 · Will pivot work?

*Predict*

> Will `listings.pivot(index="district", columns="room_type", values="price")` work? Answer first, with the reason, then run it.

*Source: course exercise.*

*Your answer:*

### 4.6 · Superhosts by district

*Pairs*

> On `listings_berlin`: what share of the listings in each district belongs to a superhost (`host_is_superhost`)? Which district has the highest share? Sort the result.

*Source: course exercise.*

### 4.7 · *Stretch:* Availability in a long table

*Code*

> `listings_berlin` has the columns `availability_30`, `availability_90` and `availability_365`. Melt them into one long table with the columns `id | window | days_available`, then compute the median per window with `groupby`.

*Source: course exercise.*

## Debrief

- Compare two answers to 4.1: one with `normalize="index"` and one with `normalize="columns"`. Both give a number for Mitte; only one answers the question. Ask which way the table sums to 1.
- In 4.4, look for solutions that used `pivot`: they fail on Mitte's two entire homes. The error message is the lesson.
- Next: the plotting block, where the long table of Step 5 becomes a chart.

## Further reading

- pandas development team, [Reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html): `melt`, `pivot`, `pivot_table`, `crosstab`, `stack` and `unstack`.
- McKinney, [Python for Data Analysis, 3rd ed., chapter 8: Data wrangling: join, combine and reshape](https://wesmckinney.com/book/data-wrangling.html), open access.
- VanderPlas, [Python Data Science Handbook, Pivot tables](https://jakevdp.github.io/PythonDataScienceHandbook/03.09-pivot-tables.html).